# Evaluating Context Quality with Context Precision

In advanced Retrieval-Augmented Generation (RAG) systems, the quality of the retrieved context is often the single most critical bottleneck. Even the most sophisticated Large Language Model (LLM) cannot generate accurate answers if the provided source material is irrelevant, incomplete, or poorly ranked. This notebook introduces **Context Precision**, a vital metric for rigorously evaluating the retrieval component of your RAG pipeline.

Context Precision quantifies how relevant the retrieved chunks are to both the user's query and the established ground truth (the reference answer). It moves beyond simply checking if *an* answer is generated, focusing instead on whether the system provided the *right information*. For developers building complex agents using frameworks like LangGraph, understanding context precision is paramount. If a retrieval step fails—for instance, by returning irrelevant documents or burying key facts deep within noise—the entire graph execution will fail, regardless of how robust the subsequent LLM calls are.

By mastering this metric, you learn to systematically diagnose and improve your vector store indexing, chunking strategies, and retriever configurations. This knowledge is essential for moving from a proof-of-concept RAG system to a production-grade, reliable application that can guarantee high factual accuracy by ensuring the context provided to the LLM is maximally relevant and efficiently ranked.

### Learning Objectives

Upon completing this notebook, you will be able to:

*   **Define Context Precision:** Understand what Context Precision measures and why it is a critical evaluation metric for RAG systems.
*   **Analyze Retrieval Failures:** Interpret how irrelevant or poorly ranked context chunks (as demonstrated in Example 3) negatively impact the perceived quality of the retrieval step.
*   **Implement Evaluation Metrics:** Utilize the `ragas` library to programmatically calculate Context Precision scores, moving beyond manual inspection to quantitative evaluation.
*   **Improve RAG Pipelines:** Apply knowledge of context relevance to diagnose and improve weak points in a multi-step RAG workflow (e.g., optimizing chunk size or improving embedding models).


### Context Precision Scorer Initialization

This cell initializes the necessary components for calculating context precision. It sets up an asynchronous OpenAI client, instantiates a language model using `llm_factory`, and finally creates the `ContextPrecision` scorer object, which will use the specified LLM to evaluate how well retrieved context supports the generated answer.


In [ ]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import ContextPrecision

# Initialize the asynchronous OpenAI client for API calls
client = AsyncOpenAI()
# Instantiate a language model (e.g., 'gpt-5-mini') using the factory pattern
llm = llm_factory("gpt-5-mini", client=client)

# Create the ContextPrecision scorer object, passing the initialized LLM to it
scorer = ContextPrecision(llm=llm)


d:\rag-evaluation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Context Precision Scoring

This cell demonstrates the use of a context scoring mechanism (`scorer.ascore`) to evaluate how well retrieved documents support a given query and reference answer. It calculates 'Context Precision,' which measures if the most relevant information is present among the retrieved chunks, even when irrelevant noise follows.


In [ ]:
# Example 1 : Relevant context is first but two irrelevant chunks follow
result = await scorer.ascore(
    user_input="What is photosynthesis?",
    reference="Photosynthesis is the process by which plants use sunlight, water, and carbon dioxide to produce glucose and oxygen.",
    retrieved_contexts=[
        "Photosynthesis is the biological process through which plants convert sunlight and carbon dioxide into glucose and oxygen.", # Relevant chunk 1 (Good match)
        "Plants require water and minerals from the soil for growth.", # Irrelevant chunk 2
        "The Amazon rainforest is home to thousands of plant species."
    ]
)
print(f"Context Precision Score: {result.value}")


Context Precision Score: 0.9999999999


This cell simply displays the final value of the `result` variable, which holds the output generated by the preceding LangGraph execution. It is necessary to view and inspect the structured or processed data returned by the advanced RAG pipeline.


In [5]:
result # Displays the final output stored in the 'result' variable.


MetricResult(value=0.9999999999)

### Context Precision Scoring

This cell demonstrates the calculation of 'Context Precision,' a crucial metric that evaluates how relevant the retrieved documents are to the user's query and the provided reference answer. It uses an asynchronous scoring function (`scorer.ascore`) to quantify the proportion of useful information within the context set.


In [ ]:
# Example 2 : All retrieved contexts are relevant and support the reference answer
result = await scorer.ascore(
    user_input="Where is the Eiffel Tower located?",
    reference="The Eiffel Tower is located in Paris, France.",
    retrieved_contexts=[
        "The Eiffel Tower is located on the Champ de Mars in Paris, France.",
        "The Eiffel Tower was built in 1889 and stands 330 meters tall in Paris."
    ]
)
print(f"Context Precision Score: {result.value}")


Context Precision Score: 0.99999999995


### Context Precision Scoring

This cell demonstrates the calculation of 'Context Precision,' a crucial metric for evaluating RAG systems. It uses an asynchronous scorer to assess how many of the retrieved context chunks are actually relevant to the user's query, even when irrelevant noise is present.


In [7]:
# Example 3 : Two completely irrelevant chunks are ranked before the one relevant chunk
result = await scorer.ascore(
    user_input="What is the boiling point of water?",
    reference="Water boils at 100 degrees Celsius (212 degrees Fahrenheit) at sea level.",
    retrieved_contexts=[
        "The capital of France is Paris, a major European city known for the Eiffel Tower.", # Irrelevant chunk 1
        "The density of water is 1 gram per cubic centimeter at 4 degrees Celsius.", # Irrelevant chunk 2
        "Water boils at 100 degrees Celsius (212 degrees Fahrenheit) at standard atmospheric pressure." # Relevant chunk
    ]
)
print(f"Context Precision Score: {result.value}")


Context Precision Score: 0.3333333333
